# Exploring the football dataset

A tour of what the pipeline produces and why some of the modelling choices
were made. Run `scripts/build_dataset.py` and `scripts/build_features.py`
first.

Nothing here trains a production model — it is for looking at the data.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from football_predictor.pipeline import load_dataset, summarise
from football_predictor.config import data_dir

matches = load_dataset()
print(f"{len(matches):,} matches, {matches['season'].min()} to {matches['season'].max()}")
summarise(matches)

## Home advantage, and how it has moved

Home advantage is real, varies by league, and collapsed during the 2020/21 crowd ban.

In [ ]:
home_win = matches.assign(
    home_win=(matches["home_goals"] > matches["away_goals"]).astype(float)
)
by_league = home_win.groupby("competition")["home_win"].mean().sort_values(ascending=False)
print(by_league.to_string())

by_season = (
    home_win[home_win["competition"] != "UEFA_UCL"]
    .groupby("season")["home_win"].mean()
)
print("\nhome-win rate, recent seasons:")
print(by_season.tail(10).to_string())

## The first half is not half the match

This is the measurement behind modelling the halves separately rather than
splitting a full-time expectation in two.

In [ ]:
with_ht = matches[matches["ht_home_goals"].notna()]
first = (with_ht["ht_home_goals"] + with_ht["ht_away_goals"]).mean()
total = (with_ht["home_goals"] + with_ht["away_goals"]).mean()
print(f"first-half goals per match : {first:.3f}")
print(f"full-time goals per match  : {total:.3f}")
print(f"first-half share           : {first / total:.1%}")
print(f"second-half share          : {1 - first / total:.1%}")

by_comp = with_ht.groupby("competition").apply(
    lambda g: (g["ht_home_goals"] + g["ht_away_goals"]).mean()
    / (g["home_goals"] + g["away_goals"]).mean(),
    include_groups=False,
)
print("\nfirst-half share by competition:")
print(by_comp.round(4).to_string())

## Why Dixon-Coles exists

Independent Poisson misprices exactly four scorelines. Comparing observed
frequencies against what a Poisson fitted to the same means would predict
shows the gap the `rho` correction closes.

In [ ]:
from scipy.stats import poisson

league = matches[matches["competition"] == "ENG_PL"]
lam = league["home_goals"].mean()
mu = league["away_goals"].mean()

rows = []
for h, a in [(0, 0), (1, 0), (0, 1), (1, 1), (2, 1), (2, 0)]:
    observed = ((league["home_goals"] == h) & (league["away_goals"] == a)).mean()
    expected = poisson.pmf(h, lam) * poisson.pmf(a, mu)
    rows.append({"score": f"{h}-{a}", "observed": observed,
                 "independent_poisson": expected,
                 "ratio": observed / expected})
pd.DataFrame(rows).round(4)

On this data the clearest signal is 0-0, which occurs about 16% more often than an independent Poisson with the same means predicts, while 0-1 occurs about 7% less often. That is the dependence between the two scores at low scorelines &mdash; exactly the four cells Dixon-Coles rescales with `rho`. Note this pools thirty seasons and both eras of scoring rates; the fitted model applies time decay and estimates `rho` per competition, where it comes out at roughly -0.15 in the Bundesliga and near zero in the Premier League.

## Feature coverage

What is actually available, and where the gaps are.

In [ ]:
features = pd.read_parquet(data_dir() / "processed" / "features.parquet")
from football_predictor.features.builder import feature_columns

columns = feature_columns(features)
coverage = features[columns].notna().mean().sort_values()
print(f"{len(columns)} numeric features")
print("\nleast covered:")
print(coverage.head(10).round(3).to_string())
print("\nfully covered:", int((coverage == 1.0).sum()))

## Fitting a goal model as of a date

The cutoff is applied inside the model, so a fit is reproducible from the date alone.

In [ ]:
from football_predictor.models.dixon_coles import DixonColesModel, GoalModelConfig

model = DixonColesModel(GoalModelConfig.from_config("dixon_coles")).fit(
    matches[matches["competition"] == "ENG_PL"],
    reference_date=pd.Timestamp("2025-08-01"),
)
print(f"rho = {model.rho:+.4f}   home advantage = {model.home_advantage:.4f}")
print(f"fitted on {model.n_matches} matches, converged={model.converged}")
model.team_strengths().head(10)

In [ ]:
matrix = model.score_matrix("Liverpool", "Arsenal")
print("expected goals: %.2f - %.2f" % (matrix.expected_home_goals, matrix.expected_away_goals))
print("HUB:", {k: round(v, 3) for k, v in matrix.result_probabilities().items()})
print("BTTS yes: %.3f" % matrix.btts()["yes"])
print("over 2.5: %.3f" % matrix.over_under()["2.5"]["over"])
print("most likely:", [s["score"] for s in matrix.top_scores(4)])

## rho and home advantage differ by league

One of the reasons the models are competition-aware rather than pooled.

In [ ]:
rows = []
for code in ["ENG_PL", "ESP_LL", "GER_BL", "ITA_SA", "FRA_L1"]:
    fitted = DixonColesModel(GoalModelConfig.from_config("dixon_coles")).fit(
        matches[matches["competition"] == code],
        reference_date=pd.Timestamp("2025-08-01"),
    )
    rows.append({"competition": code, "rho": fitted.rho,
                 "home_advantage": fitted.home_advantage,
                 "goals_per_team": fitted.baseline_rate})
pd.DataFrame(rows).round(4)